# Lyric Engine - Kaggle Training Notebook

Run this notebook top to bottom with `Run All`.

**Repo:** https://github.com/SMXFREEZE/lyric-engine
**Branch:** `codex/emotional-specificity-module`
**Recommended runtime:** GPU T4 x2

## What this notebook does
1. Checks GPU and environment
2. Clones the repo and switches to the correct branch
3. Installs core dependencies with optional audio fallback
4. Loads optional Kaggle secrets
5. Runs a smoke test on the branch
6. Automatically finds an uploaded JSONL lyrics dataset anywhere under `/kaggle/input`
7. Normalizes it into `data/raw/all_songs.jsonl` and `data/raw/<genre>.jsonl`
8. Builds style vectors
9. Builds a viral conditioning vector with a safe fallback
10. Verifies the pipeline on a real sample
11. Trains the genre adapter
12. Tests generation and metacognitive outputs
13. Optionally generates audio if audio packages installed
14. Prints checkpoint summary

## Recommended Kaggle setup
- Upload a `.jsonl` lyrics dataset before running. Good filenames are `songs.jsonl`, `trap.jsonl`, or `all_songs.jsonl`.
- Optional Kaggle secrets:
  - `HF_TOKEN`
  - `VAGALUME_API_KEY`
  - `GENIUS_TOKEN`


In [ ]:
# -- 1. Check GPU --
import torch

if not torch.cuda.is_available():
    raise SystemExit('Enable GPU first: Settings -> Accelerator -> GPU T4 x2')

n_gpus = torch.cuda.device_count()
total_vram = 0.0
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    total_vram += vram
    print(f'GPU {i}: {name} ({vram:.1f} GB)')

print(f'Total VRAM: {total_vram:.1f} GB')
print(f'GPU count : {n_gpus}')
print('Mode      : fp16' if total_vram >= 20 else 'Mode      : 4-bit fallback')

import subprocess
subprocess.check_call(['nvidia-smi'])


In [ ]:
# -- 2. Clone / update repo --
import os
import sys
import subprocess

REPO = 'https://github.com/SMXFREEZE/lyric-engine'
BRANCH = 'codex/emotional-specificity-module'
DEST = '/kaggle/working/lyric-engine'
CHECKPOINT_DIR = '/kaggle/working/checkpoints'

if not os.path.exists(DEST):
    subprocess.check_call(['git', 'clone', '-b', BRANCH, REPO, DEST])
else:
    subprocess.check_call(['git', '-C', DEST, 'fetch', 'origin'])
    subprocess.check_call(['git', '-C', DEST, 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', DEST, 'pull', 'origin', BRANCH])

os.chdir(DEST)
if DEST not in sys.path:
    sys.path.insert(0, DEST)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/style_vectors', exist_ok=True)
os.makedirs('data/charts', exist_ok=True)
os.makedirs('data/crawl', exist_ok=True)

print('Repo exists:', os.path.exists(DEST))
print('CWD:', os.getcwd())
subprocess.check_call(['git', '-C', DEST, 'branch', '--show-current'])
subprocess.check_call(['git', '-C', DEST, 'log', '--oneline', '-3'])


In [ ]:
# -- 3. Install dependencies --
import subprocess
import sys

os.chdir(DEST)


def pip_install(packages, label):
    print(f'Installing {label}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
    print(f'{label} installed.')


core_packages = [
    'transformers', 'peft', 'accelerate', 'bitsandbytes', 'trl',
    'datasets', 'lyricsgenius', 'billboard.py',
    'pronouncing', 'phonemizer', 'sentence-transformers', 'scikit-learn', 'nltk', 'textblob',
    'fastapi', 'uvicorn', 'rich', 'typer', 'python-dotenv',
    'numpy', 'pandas', 'tqdm', 'httpx',
    'pydub', 'soundfile', 'scipy',
]
optional_audio_packages = ['audiocraft', 'suno-bark']

pip_install(core_packages, 'core dependencies')

audio_ready = True
try:
    pip_install(optional_audio_packages, 'audio dependencies')
except Exception as exc:
    audio_ready = False
    print(f'Audio dependencies failed: {exc}')
    print('Continuing without full audio generation support for this session.')

print('Audio ready:', audio_ready)


In [ ]:
# -- 4. Config and secrets --
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()


def maybe_secret(name: str) -> str:
    try:
        return secrets.get_secret(name)
    except Exception:
        return ''


GENIUS_TOKEN = maybe_secret('GENIUS_TOKEN')
VAGALUME_API_KEY = maybe_secret('VAGALUME_API_KEY')
HF_TOKEN = maybe_secret('HF_TOKEN')

if GENIUS_TOKEN:
    os.environ['GENIUS_TOKEN'] = GENIUS_TOKEN
if VAGALUME_API_KEY:
    os.environ['VAGALUME_API_KEY'] = VAGALUME_API_KEY
if HF_TOKEN:
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

GENRE = 'trap'
BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'
HF_DATASET_REPO = 'SMXFREEZE/lyric-engine-data'
RUN_LIVE_COLLECTION = False
SCRAPE_CHARTS = True
MAX_SONGS_PER_ARTIST = 50
MIN_STYLE_SONGS = 2
EPOCHS = 3
LR = 2e-4

USE_4BIT = total_vram < 20
MUSICGEN_SIZE = 'large' if total_vram >= 20 else 'small'
LORA_RANK = 64 if total_vram >= 20 else 16
BATCH_SIZE = 4 if total_vram >= 20 else 2
GRAD_ACCUM = 8 if total_vram >= 20 else 16
RUN_STAGE1 = False

os.environ['HF_DATASET_REPO'] = HF_DATASET_REPO

print('Base model   :', BASE_MODEL)
print('Genre        :', GENRE)
print('4-bit        :', USE_4BIT)
print('MusicGen size:', MUSICGEN_SIZE)
print('LoRA rank    :', LORA_RANK)
print('Eff batch    :', BATCH_SIZE * GRAD_ACCUM)
print('Run stage 1  :', RUN_STAGE1)
print('Genius token :', 'yes' if GENIUS_TOKEN else 'optional / missing')
print('Vagalume key :', 'yes' if VAGALUME_API_KEY else 'optional / missing')
print('HF token     :', 'yes' if HF_TOKEN else 'optional / missing')


In [ ]:
# -- 5. Smoke validation --
import subprocess
import sys

os.chdir(DEST)
print('Running smoke test...')
subprocess.check_call([sys.executable, 'scripts/smoke_test.py'])
print('Smoke test passed.')


In [ ]:
# -- 6. Find and normalize lyrics dataset --
import json
from collections import Counter
from pathlib import Path

os.chdir(DEST)

raw_dir = Path('data/raw')
crawl_dir = Path('data/crawl')
raw_dir.mkdir(parents=True, exist_ok=True)
crawl_dir.mkdir(parents=True, exist_ok=True)

genre_path = raw_dir / f'{GENRE}.jsonl'
all_path = raw_dir / 'all_songs.jsonl'
import_path = crawl_dir / 'imported.jsonl'


def discover_jsonl_files(root: str) -> list[Path]:
    root_path = Path(root)
    if not root_path.exists():
        return []
    return sorted(root_path.rglob('*.jsonl'))


def candidate_rank(path: Path):
    preferred = {
        f'{GENRE}.jsonl': 0,
        'songs.jsonl': 1,
        'all_songs.jsonl': 2,
    }
    return (preferred.get(path.name.lower(), 10), len(path.parts), str(path))


def load_jsonl_records(path: Path) -> list[dict]:
    records = []
    with open(path, 'r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return records


def normalize_records(records: list[dict], default_genre: str) -> list[dict]:
    normalized = []
    seen = set()
    for record in records:
        artist = str(record.get('artist', '')).strip()
        title = str(record.get('title', '')).strip()
        lyrics = str(record.get('lyrics', '')).strip()
        if not artist or not title or not lyrics:
            continue
        genre = str(record.get('genre') or default_genre).strip().lower()
        key = (artist.lower(), title.lower())
        if key in seen:
            continue
        seen.add(key)
        row = dict(record)
        row['artist'] = artist
        row['title'] = title
        row['lyrics'] = lyrics
        row['genre'] = genre
        normalized.append(row)
    return normalized


def write_jsonl(path: Path, records: list[dict]) -> None:
    with open(path, 'w', encoding='utf-8') as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')


source_label = ''
all_records = []
input_matches = discover_jsonl_files('/kaggle/input')
preferred_matches = [p for p in input_matches if p.name.lower() in {f'{GENRE}.jsonl', 'songs.jsonl', 'all_songs.jsonl'}]

if preferred_matches:
    chosen = sorted(preferred_matches, key=candidate_rank)[0]
    source_label = str(chosen)
    all_records = normalize_records(load_jsonl_records(chosen), GENRE)
    print('Using Kaggle dataset:', source_label)
elif HF_TOKEN:
    try:
        from datasets import load_dataset
        ds = load_dataset(HF_DATASET_REPO, split='train', token=HF_TOKEN)
        all_records = normalize_records([dict(row) for row in ds], GENRE)
        source_label = f'HF:{HF_DATASET_REPO}'
        print('Using HuggingFace dataset:', source_label)
    except Exception as exc:
        print('HF load failed:', exc)

if not all_records and RUN_LIVE_COLLECTION:
    from src.data.scraper import run_scrape
    print('No uploaded dataset found. Running live multi-provider scrape...')
    scraped = run_scrape(
        genres=[GENRE],
        max_songs_per_artist=MAX_SONGS_PER_ARTIST,
        out_dir='data/raw',
        token=GENIUS_TOKEN,
    )
    all_records = normalize_records(scraped, GENRE)
    source_label = 'live scrape'

if not all_records:
    print('Visible JSONL files under /kaggle/input:')
    for path in input_matches[:50]:
        print(' -', path)
    raise SystemExit('No usable lyrics dataset found. Upload a JSONL dataset before using Run All.')

genre_records = [row for row in all_records if row['genre'] == GENRE]
if not genre_records:
    genre_records = [dict(row, genre=GENRE) for row in all_records]

write_jsonl(import_path, all_records)
write_jsonl(all_path, all_records)
write_jsonl(genre_path, genre_records)

DATASET_TOTAL_ROWS = len(all_records)
DATASET_GENRE_ROWS = len(genre_records)
DATASET_ARTISTS = len({row['artist'] for row in genre_records})
DATASET_GENRES = Counter(row['genre'] for row in all_records)

print('Dataset source  :', source_label)
print('All songs rows  :', DATASET_TOTAL_ROWS)
print('Genre rows      :', DATASET_GENRE_ROWS)
print('Genre artists   :', DATASET_ARTISTS)
print('Genre breakdown :', dict(DATASET_GENRES.most_common(10)))
print('Saved all songs :', all_path)
print('Saved genre file:', genre_path)

if DATASET_GENRE_ROWS == 0:
    raise SystemExit(f'No rows available for genre {GENRE}.')


In [ ]:
# -- 7. Build style vectors --
from src.data.style_extractor import build_style_vectors

os.chdir(DEST)
print('Building artist style vectors...')
style_vectors = build_style_vectors(
    jsonl_path=f'data/raw/{GENRE}.jsonl',
    out_dir='data/style_vectors',
    min_songs=MIN_STYLE_SONGS,
)
print(f'Style vectors built for {len(style_vectors)} artists')


In [ ]:
# -- 8. Viral DNA conditioning --
import datetime
import json
import numpy as np

os.chdir(DEST)

VIRAL_VECTOR_PATH = 'data/viral_conditioning.npy'
VIRAL_REPORT_PATH = 'data/viral_report.json'
viral_vec = np.zeros(32, dtype=np.float32)
viral_report = {'status': 'fallback', 'reason': 'not run'}

if SCRAPE_CHARTS:
    try:
        from src.data.chart_scraper import scrape_all_charts, compute_viral_scores
        from src.data.viral_analyzer import run_analysis, build_conditioning_vector

        records = scrape_all_charts(out_dir='data/charts')
        viral_scores = compute_viral_scores(records)
        week = datetime.datetime.utcnow().strftime('%Y-%W')
        viral_report = run_analysis(week=week, charts_dir='data/charts')
        viral_vec = build_conditioning_vector(
            genre=GENRE,
            region=None,
            viral_scores=viral_scores,
            records=records,
        )
        if viral_vec.shape[0] != 32:
            fixed = np.zeros(32, dtype=np.float32)
            fixed[:min(32, viral_vec.shape[0])] = viral_vec[:32]
            viral_vec = fixed
    except Exception as exc:
        viral_report = {'status': 'fallback', 'reason': str(exc)}
        print('Viral DNA fallback:', exc)
else:
    viral_report = {'status': 'skipped', 'reason': 'SCRAPE_CHARTS=False'}

np.save(VIRAL_VECTOR_PATH, viral_vec)
with open(VIRAL_REPORT_PATH, 'w', encoding='utf-8') as handle:
    json.dump(viral_report, handle, ensure_ascii=False, indent=2)

print('Viral vector shape:', viral_vec.shape)
print('Viral vector norm :', float(np.linalg.norm(viral_vec)))
print('Viral report path :', VIRAL_REPORT_PATH)


In [ ]:
# -- 9. Verify pipeline --
import json
from src.data.phoneme_annotator import annotate_line
from src.data.rhyme_labeler import detect_scheme
from src.data.valence_scorer import score_lyrics

os.chdir(DEST)

with open(f'data/raw/{GENRE}.jsonl', 'r', encoding='utf-8') as handle:
    sample = json.loads(handle.readline())

lines = [line for line in sample['lyrics'].splitlines() if line.strip()][:8]
lyrics_block = '\n'.join(lines)
scheme = detect_scheme(lines)

print('Artist:', sample['artist'])
print('Title :', sample['title'])
print('Rows  :', DATASET_GENRE_ROWS)
print('Artists in genre:', DATASET_ARTISTS)
print('Rhyme scheme :', scheme['scheme_str'], scheme['scheme_type'])
print('Rhyme density:', scheme['rhyme_density'])
print()
for em in score_lyrics(lyrics_block):
    ann = annotate_line(em.text)
    print(f'[{em.valence:+.2f}v {em.arousal:.2f}a] [{ann.total_syllables:2d}syl] {em.text[:80]}')
print() 
print('Pipeline OK')


In [ ]:
# -- 10. Train LoRA --
import numpy as np
from src.training.sft import train_sft

os.chdir(DEST)

viral_vec = np.load('data/viral_conditioning.npy') if os.path.exists('data/viral_conditioning.npy') else np.zeros(32, dtype=np.float32)
print('Viral conditioning norm:', float(np.linalg.norm(viral_vec)))

all_data_path = 'data/raw/all_songs.jsonl'
genre_data_path = f'data/raw/{GENRE}.jsonl'

if RUN_STAGE1 and DATASET_TOTAL_ROWS >= 2000 and len(DATASET_GENRES) > 1:
    print('Running stage 1 on all songs...')
    train_sft(
        stage=1,
        data_path=all_data_path,
        base_model=BASE_MODEL,
        output_dir=CHECKPOINT_DIR,
        batch_size=BATCH_SIZE,
        grad_accum_steps=GRAD_ACCUM,
        epochs=1,
        lr=LR,
        lora_rank=64,
        alpha=128,
        use_4bit=USE_4BIT,
        style_vec_dir='data/style_vectors',
        viral_conditioning=viral_vec,
    )
else:
    print('Skipping stage 1 (single-genre or small dataset).')

print(f'Running stage 2 for genre {GENRE}...')
train_sft(
    stage=2,
    genre=GENRE,
    data_path=genre_data_path,
    base_model=BASE_MODEL,
    output_dir=CHECKPOINT_DIR,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM,
    epochs=EPOCHS,
    lr=LR,
    lora_rank=LORA_RANK,
    alpha=LORA_RANK * 2,
    use_4bit=USE_4BIT,
    style_vec_dir='data/style_vectors',
    viral_conditioning=viral_vec,
)

FINAL_CKPT = f'{CHECKPOINT_DIR}/genre_{GENRE}/epoch_{EPOCHS}'
print('Final checkpoint:', FINAL_CKPT)


In [ ]:
# -- 11. Test generation + metacognitive outputs --
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer as HFTok
from src.inference.engine import LyricsEngine, SongMemory

os.chdir(DEST)

ckpt_path = FINAL_CKPT
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Loading checkpoint from', ckpt_path)
tok = HFTok.from_pretrained(ckpt_path)
tok.pad_token = tok.eos_token

base_kwargs = {'device_map': 'auto'} if torch.cuda.is_available() else {}
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **base_kwargs)
model = PeftModel.from_pretrained(base, ckpt_path)
engine = LyricsEngine(model, tok, device=device, beam_size=5)

memory = SongMemory(
    genre=GENRE,
    rhyme_scheme='AABB',
    target_syllables=10,
)

candidates = engine.generate_line(memory, top_n=3)
print('Candidates:', len(candidates))
for idx, candidate in enumerate(candidates, 1):
    print(f'[{idx}] {candidate.text}')
    print(f'    total={candidate.total_score:.3f} phonetic={candidate.phonetic_score:.3f} novelty={candidate.novelty_score:.3f}')

if candidates:
    memory.add_line(candidates[0].text, section='verse1')

analysis = engine.analyze_song(memory)
print() 
print('Metacognition keys:', sorted(analysis.get('metacognition', {}).keys()))
print('Last workspace keys:', sorted(analysis.get('last_workspace', {}).keys()) if analysis.get('last_workspace') else [])
print('hot_trace:', analysis.get('last_workspace', {}).get('hot_trace', {}))


In [ ]:
# -- 12. Optional audio generation --
os.chdir(DEST)

if not globals().get('audio_ready', False):
    print('Skipping audio generation because optional audio dependencies are not available.')
else:
    from src.audio.song_assembler import SongAssembler, SongSpec

    generated_lines = [c.text for c in candidates[:8]] if 'candidates' in globals() else []
    if not generated_lines:
        raise SystemExit('No generated lyrics available. Run the generation cell first.')

    spec = SongSpec(
        title='Kaggle Demo Song',
        genre=GENRE,
        bpm=140,
        mood='hype',
        language='english',
        voice_idx=0,
        sections=['verse', 'hook'],
        lyrics='\n'.join(generated_lines),
    )

    assembler = SongAssembler(musicgen_size=MUSICGEN_SIZE)
    song = assembler.generate_full_song(spec, out_dir='outputs/audio')
    print('Audio output:', song.output_path)


In [ ]:
# -- 13. Checkpoint summary --
import os

os.chdir(DEST)

ckpt = FINAL_CKPT
print('Checkpoint:', ckpt)
print()
if os.path.exists(ckpt):
    total_mb = 0.0
    for fname in sorted(os.listdir(ckpt)):
        path = os.path.join(ckpt, fname)
        if not os.path.isfile(path):
            continue
        size_mb = os.path.getsize(path) / 1e6
        total_mb += size_mb
        print(f'{fname:45s} {size_mb:6.1f} MB')
    print() 
    print(f'Total checkpoint size: {total_mb:.1f} MB')
else:
    print('Checkpoint directory not found.')

print() 
print('To download from Kaggle: open the right sidebar -> Output -> /kaggle/working/checkpoints')
